# Benchmark Metasains Psikometri: Validitas Isi & Kesepakatan Rater
### Komparasi Presisi: Aiken's V, Exact P-Value (Konvolusi Polinomial), Gwet's AC1, & Resampling Bootstrap

Notebook ini merupakan artefak komputasi pendamping resmi untuk monograf **_Jangkar & Kompas Riset Ilmiah_** (Bab 11: Presisi Tabel & Visualisasi, serta Lampiran C: Validitas Isi Aiken & Bootstrap Non-Parametrik).

**Tujuan Pembuktian Ilmiah:**
1. Menguji akurasi matematis penghitungan _exact p-value_ berbasis konvolusi polinomial diskrit $\frac{1}{c^n}\left(\sum_{k=0}^{c-1} x^k\right)^n$ tanpa asumsi aproksimasi normal.
2. Membandingkan koefisien validitas Aiken $V$ dengan koefisien kesepakatan antar-penilai terkoreksi peluang (Gwet's $AC_1$) untuk membedah dilema polarisasi bimodal.
3. Menguji selang kepercayaan 95% non-parametrik melalui 1.000 iterasi _bootstrap resampling_ pada 1.000 butir instrumen tolok ukur terbuka.

**Rujukan Metasains Ber-DOI:**
- Aiken, L. R. (1985). *Three coefficients for analyzing the reliability and validity of ratings*. Educational and Psychological Measurement, 45(1), 131–142. https://doi.org/10.1177/0013164485451012
- Ayre, C., & Scally, A. J. (2014). *Critical values for Lawshe's content validity ratio*. Measurement and Evaluation in Counseling and Development, 47(1), 79–86. https://doi.org/10.1177/0748175613513808
- Gwet, K. L. (2008). *Computing inter-rater reliability and its variance in the presence of high agreement*. British Journal of Mathematical and Statistical Psychology, 61(1), 29–48. https://doi.org/10.1348/000711006X126600
- Nuijten, M. B., et al. (2016). *The prevalence of statistical reporting errors in psychology (1985–2013)*. Behavior Research Methods, 48(4), 1205–1226. https://doi.org/10.3758/s13428-015-0664-2

In [ ]:
# [1] IMPORT MODUL STANDAR (Murni Python Standard Library & Scientific Ecosystem)
import csv
import math
import random
from pathlib import Path

# Menjamin keterulangan eksperimen (Zero Phantom Randomness)
random.seed(2026)
print("[OK] Lingkungan komputasi siap. Zero external proprietary dependencies.")

In [ ]:
# [2] IMPLEMENTASI DETERMINISTIK MOTOR PSIKOMETRI (ACM/APA PSEUDOCODE C.1 - C.3)

def calculate_aiken_v(ratings, lo=1, hi=5):
    """Koefisien Validitas Isi Aiken V (1985)"""
    n = len(ratings)
    c = hi - lo + 1
    s_obs = sum(r - lo for r in ratings)
    s_max = n * (c - 1)
    return s_obs / s_max if s_max > 0 else 0.0

def calculate_aiken_exact_p_value(ratings, lo=1, hi=5):
    """Derivasi Exact P-Value via Konvolusi Polinomial Diskrit 1D DP (Algoritma C.1)"""
    n = len(ratings)
    c = hi - lo + 1
    s_obs = sum(r - lo for r in ratings)
    s_max = n * (c - 1)
    if s_obs <= 0: return 1.0
    if s_obs >= s_max: return (1.0 / c) ** n

    dp = [1] * c
    for r in range(2, n + 1):
        cur_max = r * (c - 1)
        next_dp = [0] * (cur_max + 1)
        for k in range(c):
            for s in range(len(dp)):
                next_dp[s + k] += dp[s]
        dp = next_dp

    total_ways = c ** n
    tail_ways = sum(dp[s_obs : s_max + 1])
    return tail_ways / total_ways

def calculate_gwet_ac1(ratings, lo=1, hi=5):
    """Kesepakatan Antar-Rater Terkoreksi Peluang Gwet AC1 (Algoritma C.2)"""
    n = len(ratings)
    c = hi - lo + 1
    if n < 2: return 1.0
    counts = [0] * c
    for r in ratings:
        cat = max(0, min(c - 1, round(r) - lo))
        counts[cat] += 1

    total_pairs = (n * (n - 1)) // 2
    agree_pairs = sum((cnt * (cnt - 1)) // 2 for cnt in counts if cnt > 1)
    pa = agree_pairs / total_pairs

    pe_sum = sum((cnt / n) * (1.0 - (cnt / n)) for cnt in counts)
    pe = (1.0 / (c - 1)) * pe_sum
    return 1.0 if abs(1.0 - pe) < 1e-9 else (pa - pe) / (1.0 - pe)

def bootstrap_aiken_ci(ratings, lo=1, hi=5, n_resamples=2000, seed=42):
    """Selang Kepercayaan Non-Parametrik Bootstrap (Algoritma C.3)"""
    n = len(ratings)
    if n < 2: return (calculate_aiken_v(ratings, lo, hi), calculate_aiken_v(ratings, lo, hi))
    rng = random.Random(seed)
    boot_vs = []
    for _ in range(n_resamples):
        sample = [rng.choice(ratings) for _ in range(n)]
        boot_vs.append(calculate_aiken_v(sample, lo, hi))
    boot_vs.sort()
    idx_lo = int(0.025 * n_resamples)
    idx_hi = int(0.975 * n_resamples) - 1
    return (round(boot_vs[idx_lo], 4), round(boot_vs[idx_hi], 4))

print("[OK] Seluruh algoritma metrologi psikometri terkompilasi deterministik.")

In [ ]:
# [3] MEMUAT DAN MEMVERIFIKASI DATASET TOLOK UKUR (1.000 BUTIR)
data_path = Path("../data/psychometric_validation_benchmark.csv")
if not data_path.exists():
    # Fallback jika dijalankan di root
    data_path = Path("benchmarks/data/psychometric_validation_benchmark.csv")

benchmark_records = []
with open(data_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        row["n_raters"] = int(row["n_raters"])
        row["n_categories"] = int(row["n_categories"])
        row["ratings_list"] = [int(x) for x in row["ratings"].split(";")]
        row["aiken_v"] = float(row["aiken_v"])
        row["exact_p_value"] = float(row["exact_p_value"])
        row["critical_v_05"] = float(row["critical_v_05"])
        row["gwet_ac1"] = float(row["gwet_ac1"])
        benchmark_records.append(row)

print(f"[OK] Berhasil memuat {len(benchmark_records)} butir dari {data_path.name}.")

In [ ]:
# [4] AUDIT KONKORDANSI MATEMATIS 100% (ZERO DISCREPANCY VERIFICATION)
errors = 0
for idx, item in enumerate(benchmark_records):
    v_recomputed = calculate_aiken_v(item["ratings_list"], lo=1, hi=item["n_categories"])
    p_recomputed = calculate_aiken_exact_p_value(item["ratings_list"], lo=1, hi=item["n_categories"])
    ac1_recomputed = calculate_gwet_ac1(item["ratings_list"], lo=1, hi=item["n_categories"])

    if abs(v_recomputed - item["aiken_v"]) > 1e-4:
        print(f"Discrepancy in V at {item['item_id']}: {v_recomputed} vs {item['aiken_v']}")
        errors += 1
    if abs(p_recomputed - item["exact_p_value"]) > 1e-4:
        print(f"Discrepancy in Exact p at {item['item_id']}: {p_recomputed} vs {item['exact_p_value']}")
        errors += 1
    if abs(ac1_recomputed - item["gwet_ac1"]) > 1e-4:
        print(f"Discrepancy in AC1 at {item['item_id']}: {ac1_recomputed} vs {item['gwet_ac1']}")
        errors += 1

if errors == 0:
    print(f"[VERIFIED] 100% Konkordansi Sempurna ({len(benchmark_records)} butir terverifikasi identik terhadap derivasi analitis)!")
else:
    print(f"[FAIL] Ditemukan {errors} diskrepansi numerik.")

In [ ]:
# [5] ANALISIS STATISTIK ARKETIPE & MEMBEDAH DILEMA POLARISASI (TABEL RINGKASAN)
archetype_stats = {}
for r in benchmark_records:
    arc = r["archetype"]
    if arc not in archetype_stats:
        archetype_stats[arc] = {"count": 0, "valid_count": 0, "vs": [], "ps": [], "ac1s": []}
    archetype_stats[arc]["count"] += 1
    if r["verdict"] == "VALID_SIGNIFIKAN":
        archetype_stats[arc]["valid_count"] += 1
    archetype_stats[arc]["vs"].append(r["aiken_v"])
    archetype_stats[arc]["ps"].append(r["exact_p_value"])
    archetype_stats[arc]["ac1s"].append(r["gwet_ac1"])

print(f"{'Arketipe':<24} {'N Butir':<10} {'Valid (%)':<12} {'Mean V':<10} {'Mean Exact p':<14} {'Mean AC1':<10}")
print("=" * 82)
for arc, st in archetype_stats.items():
    n_tot = st["count"]
    pct_val = (st["valid_count"] / n_tot) * 100
    mean_v = sum(st["vs"]) / n_tot
    mean_p = sum(st["ps"]) / n_tot
    mean_ac1 = sum(st["ac1s"]) / n_tot
    print(f"{arc:<24} {n_tot:<10} {pct_val:<12.1f} {mean_v:<10.4f} {mean_p:<14.6f} {mean_ac1:<10.4f}")
print("=" * 82)
print("\nTemuan Kritis Epistemik:")
print("1. Pada arketipe Polarization_Dilemma, nilai Aiken V bisa tampak moderat-tinggi namun nilai Gwet AC1 anjlok,")
print("   membuktikan bahwa konsensus rater terpecah (disagree) meskipun rata-rata agregat tampak memadai.")
print("2. Hal ini menegaskan mengapa monograf merekomendasikan triangulasi instrumen: V saja tidak cukup,")
print("   wajib didampingi Exact P-Value polinomial dan Gwet's AC1 (Invarian Metodologi Kesarjanaan).")

## Kesimpulan & Diseminasi Terbuka

Berdasarkan hasil pengujian di atas:
- **Metode Konvolusi Polinomial Diskrit** memberikan akurasi absolut $O(n^2 \cdot c)$ tanpa galat pembulatan tabel statis atau ketidakpastian simulasi stokastik.
- **Kemitraan Triad Komputasi:** Modul ini telah terintegrasi penuh di:
  1. Repositori Riset Terbuka: `https://github.com/zadevpenseo/jangkar-kompas-riset-ilmiah`
  2. Pustaka Python Mandiri: `pip install scholarly-core` (CLI: `scholarly compute validity`)
  3. Laboratorium Interaktif Web: `https://ilmify.id/lab#aiken` (Didukung Cloudflare Pages Edge Engine)